# Etapa 7 - Recomendacoes finais e plano de execucao comercial

Notebook executavel da Etapa 7. A logica canonica fica em `etapa7_recomendacoes_finais.py`; este notebook
executa o script e registra uma leitura dos outputs auditaveis gerados em `outputs/etapa7/`.

A etapa e a **sintese de decisao**: consome os artefatos das Etapas 3-6 e classifica cada par loja x SKU em
uma de quatro acoes comerciais - **comprar, reprecificar, promover/queimar estoque ou descontinuar** - com
uma acao primaria por precedencia para nao dupla contar capital ou quantidade. Os tres universos
(`REDE_COMPLETA`, `REDE_FISICA_SEM_LOJA93`, `LOJA_93_ATACADO_B2B`) ficam segregados e fecham em `REDE_COMPLETA`.

<!-- glossario-comercial -->
## 📖 Glossário rápido (para leitura não técnica)

Esta seção traduz os termos técnicos da Etapa 7 para linguagem de negócio. Quem é de áreas como
comercial, compras ou financeiro pode ler as recomendações sem precisar do detalhe técnico.

| Termo | O que significa, em linguagem comercial |
|---|---|
| **Comprar (recompor)** | Repor um item que está em falta ou perto de faltar e ainda tem venda. Vem pronto da Etapa 6 (plano de 90 dias). |
| **Reprecificar** | Rever o preço/margem de um item. Vem dos candidatos da Etapa 5. |
| **Promover / queima de estoque** | Acelerar a saída de um item que está **encalhado** (muito estoque para pouca venda), com desconto controlado, para liberar capital. |
| **Descontinuar** | Tirar do sortimento um item **parado** (sem venda no período) com estoque em capital imobilizado — depois de validar que não é sazonal nem campeão. |
| **Capital imobilizado** | Dinheiro "preso" em estoque parado (estoque × custo). Só é calculado quando há custo conhecido do item. |
| **Valor de encalhe** | O mesmo conceito, aplicado ao estoque em excesso que ainda gira (candidato a promoção). |
| **Ação primária** | Cada item pode disparar mais de um sinal. Para não contar o mesmo dinheiro duas vezes, ele recebe **uma** ação principal, seguindo a ordem: descontinuar > promover > reprecificar > comprar. |
| **Sinal de margem auditável × sinal de preço** | No repricing, separamos o que é **problema de margem de verdade** (item com custo conhecido e margem baixa/negativa) do que é **sinal de preço/desconto** (com ou sem custo; sem sinal de margem auditável, não prova margem baixa). |
| **Prioridade ALTA/MÉDIA/BAIXA** | Ordem de execução dentro de cada ação: ALTA = agir primeiro (os ~10% mais urgentes). |
| **Guarda-corpo do campeão** | Um item **curva A** (grande em receita histórica) parado numa loja **não** é descontinuado; é enviado para escoar/transferir. Evita cortar um produto importante só por giro recente baixo. |
| **Rede física × atacado/B2B (Loja 93)** | A Loja 93 é atacado, com comportamento muito diferente do varejo. Fica sempre em escopo separado; `REDE_COMPLETA` é a soma reconciliada dos dois. |

> **Importante:** valor financeiro (capital, encalhe, investimento) só existe para os itens com **custo conhecido**
> (cobertura de custo da Etapa 5, ~16% da receita). "Sem custo" aparece como *não avaliado* (vazio), **nunca como zero**.

## Execucao canonica

Toda metrica importante e persistida em CSV ou Markdown. O notebook deve ser lido como registro de execucao
e leitura dos arquivos finais, nao como fonte paralela de regra de negocio.

In [1]:
from pathlib import Path
import runpy

candidatos = [Path("etapa7_recomendacoes_finais.py"), Path("notebooks/etapa7_recomendacoes_finais.py")]
script = next(p for p in candidatos if p.exists())
_ = runpy.run_path(str(script), run_name="__main__")

Carregando base da Etapa 6 e candidatos de repricing da Etapa 5...


Classificando acoes (promover/descontinuar/reprecificar/comprar)...
Agregando por universo, categoria e loja...


Validando e construindo autoaudit...
Salvando arquivos auditaveis...



--- Destaques Etapa 7 (rede fisica) ---
  DESCONTINUAR :    434 pares | valor conhecido R$ 0,4M
  PROMOVER     :    752 pares | valor conhecido R$ 6,7M
  REPRECIFICAR :    956 pares | valor conhecido nao aplicavel
  COMPRAR      : 19,126 pares | valor conhecido R$ 4,6M
  Loja 93 acionaveis: 345
  Validacoes OK: 23/23

[OK] Arquivos salvos em outputs/etapa7/
  outputs\etapa7\autoaudit_etapa7.csv
  outputs\etapa7\documentacao_tecnica_etapa7.md
  outputs\etapa7\priorizacao_acoes.csv
  outputs\etapa7\recomendacoes_acao_universo.csv
  outputs\etapa7\recomendacoes_categoria_n1.csv
  outputs\etapa7\recomendacoes_lojas.csv
  outputs\etapa7\recomendacoes_melhoria.csv
  outputs\etapa7\recomendacoes_sku_loja.csv
  outputs\etapa7\reprecificacao_candidatos.csv
  outputs\etapa7\resumo_etapa7.md
  outputs\etapa7\validacoes_etapa7.csv


## KPIs por acao e universo

Pares, SKUs e valor financeiro conhecido por acao primaria, lidos de
`outputs/etapa7/recomendacoes_acao_universo.csv`. `VALOR_FINANCEIRO_CONHECIDO` cobre so os itens com custo.

In [2]:
import pandas as pd

out = Path("outputs/etapa7") if Path("outputs/etapa7").exists() else Path("../outputs/etapa7")
acao_univ = pd.read_csv(out / "recomendacoes_acao_universo.csv", encoding="utf-8-sig")
acao_univ[[
    "UNIVERSO", "ACAO_PRIMARIA", "PARES", "SKUS",
    "PARES_COM_VALOR", "VALOR_FINANCEIRO_CONHECIDO", "COBERTURA_VALOR_PCT",
]]

,UNIVERSO,ACAO_PRIMARIA,PARES,SKUS,PARES_COM_VALOR,VALOR_FINANCEIRO_CONHECIDO,COBERTURA_VALOR_PCT
0,LOJA_93_ATACADO_B2B,DESCONTINUAR,29,29,0,NaN,0.000000
1,LOJA_93_ATACADO_B2B,PROMOVER,20,20,1,2.387492e+04,5.000000
2,LOJA_93_ATACADO_B2B,REPRECIFICAR,33,33,0,NaN,0.000000
3,LOJA_93_ATACADO_B2B,COMPRAR,263,263,12,1.264659e+06,4.562738
4,REDE_COMPLETA,DESCONTINUAR,463,357,73,3.729238e+05,15.766739
5,REDE_COMPLETA,PROMOVER,772,584,187,6.748738e+06,24.222798
6,REDE_COMPLETA,REPRECIFICAR,989,432,0,NaN,0.000000
7,REDE_COMPLETA,COMPRAR,19389,2712,1867,5.885046e+06,9.629171
8,REDE_FISICA_SEM_LOJA93,DESCONTINUAR,434,339,73,3.729238e+05,16.820276
9,REDE_FISICA_SEM_LOJA93,PROMOVER,752,573,186,6.724863e+06,24.734043


## Fila priorizada de execucao (topo da rede fisica)

Ordenada por banda de prioridade, precedencia da acao e urgencia. Cada par aparece uma unica vez.

In [3]:
fila = pd.read_csv(out / "priorizacao_acoes.csv", encoding="utf-8-sig", low_memory=False)
colunas = ["RANK_EXECUCAO", "FAIXA_PRIORIDADE", "ACAO_PRIMARIA", "COD_EMPRESA", "CODIGO",
           "DESCRICAO", "NIVEL_1", "VALOR_ACAO_PRIMARIA"]
fila[fila["UNIVERSO_OPERACIONAL"] == "REDE_FISICA_SEM_LOJA93"][colunas].head(15)

,RANK_EXECUCAO,FAIXA_PRIORIDADE,ACAO_PRIMARIA,COD_EMPRESA,CODIGO,DESCRICAO,NIVEL_1,VALOR_ACAO_PRIMARIA
345,1,ALTA,DESCONTINUAR,3,467256,VENT.COL 40CM 3V VTX-40C PT/AZ M220,D - ELETROS,49716.60000
346,2,ALTA,DESCONTINUAR,3,463806,SANDUICH.ELET.GRILL SN-01 PT M220,D - ELETROS,42787.29600
347,3,ALTA,DESCONTINUAR,3,463808,CAFET.ELET.18 XIC. C-30-18X PT M220,D - ELETROS,31533.46560
348,4,ALTA,DESCONTINUAR,2,467860,COLCHAO VISCO FIT 138X25X188,O - MOVEIS,23988.33000
349,5,ALTA,DESCONTINUAR,3,407852,ESPREM.FRUTAS 30W E-02 PT M220,D - ELETROS,21627.14400
350,6,ALTA,DESCONTINUAR,92,478705,"PORCEL.A 58X58 ESM.BOHE.ANT.AC1,68M2",C - PISOS E REVESTIMENTOS,21307.82976
351,7,ALTA,DESCONTINUAR,2,467861,COLCHAO VISCO FIT 158X25X188,O - MOVEIS,14410.34100
352,8,ALTA,DESCONTINUAR,3,443992,CAFET.ELET.20XIC. C-43-20X IX M220,D - ELETROS,12204.05760
353,9,ALTA,PROMOVER,92,479589,COND.PORT. 12000 MPH-12CRV2 M220,D - ELETROS,586014.80000
354,10,ALTA,PROMOVER,92,466539,VENT.COL.40CM 3V VF4C PT M220,D - ELETROS,389464.57880


## Repricing: sinal de margem auditavel x sinal de preco

Ponto levantado na revisao interna: separar o que e problema de margem (item **com custo**) do que e apenas
sinal de preco/lista (item **sem custo**, margem nao auditavel).

In [4]:
rep = pd.read_csv(out / "reprecificacao_candidatos.csv", encoding="utf-8-sig")
resumo_rep = pd.DataFrame({
    "candidatos_repricing": [len(rep)],
    "com_sinal_margem_auditavel": [int((rep['SINAL_MARGEM_AUDITAVEL'] == 1).sum())],
    "com_sinal_preco_lista": [int((rep['SINAL_PRECO_LISTA'] == 1).sum())],
    "apenas_preco_sem_custo": [int(((rep['SINAL_PRECO_LISTA'] == 1) & (rep['FLAG_CUSTO_VALIDO'] == 0)).sum())],
})
resumo_rep

,candidatos_repricing,com_sinal_margem_auditavel,com_sinal_preco_lista,apenas_preco_sem_custo
0,1011,41,996,895


## Validacoes

Todas as reconciliacoes numericas precisam estar `OK` antes de usar a etapa: receita fecha com a Etapa 3,
`REDE_COMPLETA` = fisica + Loja 93 por acao, sinais reconciliam com as Etapas 5/6, guarda-corpo do campeao
e financeiro nunca imputado sem custo.

In [5]:
validacoes = pd.read_csv(out / "validacoes_etapa7.csv", encoding="utf-8-sig")
print(f"Validacoes OK: {(validacoes['STATUS'] == 'OK').sum()}/{len(validacoes)}")
validacoes

Validacoes OK: 23/23


,VALIDACAO,OBSERVADO,ESPERADO,DIFERENCA,STATUS
0,Receita base vs Etapa 3 - rede completa,4.825157e+08,4.825157e+08,0.000000,OK
1,Receita base vs Etapa 3 - rede fisica,3.292348e+08,3.292348e+08,0.000002,OK
2,Receita base vs Etapa 3 - Loja 93,1.532810e+08,1.532810e+08,-0.000002,OK
3,REDE_COMPLETA = fisica + Loja 93 (pares DESCON...,4.630000e+02,4.630000e+02,0.000000,OK
4,REDE_COMPLETA = fisica + Loja 93 (pares PROMOVER),7.720000e+02,7.720000e+02,0.000000,OK
5,REDE_COMPLETA = fisica + Loja 93 (pares REPREC...,9.890000e+02,9.890000e+02,0.000000,OK
6,REDE_COMPLETA = fisica + Loja 93 (pares COMPRAR),1.938900e+04,1.938900e+04,0.000000,OK
7,Cada par acionavel tem uma unica acao primaria,1.000000e+00,1.000000e+00,0.000000,OK
8,Soma de pares por acao = total de pares aciona...,2.161300e+04,2.161300e+04,0.000000,OK
9,Sinal COMPRAR = pares com compra na Etapa 6,2.035700e+04,2.035700e+04,0.000000,OK


## Autoaudit

Revisao critica das quatro armadilhas obrigatorias e como foram tratadas.

In [6]:
pd.read_csv(out / "autoaudit_etapa7.csv", encoding="utf-8-sig")

,RISCO,COMO_PODERIA_ERRAR,CONTROLE_APLICADO,EVIDENCIA,RISCO_REMANESCENTE
0,Descontinuar campeao historico so por giro bai...,Marcar para saida um SKU curva A parado nesta ...,Curva A parado nunca entra em DESCONTINUAR; cu...,132 pares curva A parados protegidos e roteado...,Curva B/C parado ainda pode ter sazonalidade l...
1,Promover/reprecificar item como se a margem fo...,Tratar desconto/preco fora da faixa como probl...,Repricing separa SINAL_MARGEM_AUDITAVEL (tem c...,35 pares de repricing primario com sinal de ma...,Preco de lista pode estar desatualizado; desco...
2,Misturar a Loja 93 (atacado/B2B) nas recomenda...,Usar status/demanda da rede fisica para a Loja...,Universo operacional segrega a Loja 93; a base...,345 pares acionaveis sao da Loja 93 e ficam no...,Pedidos B2B sao intermitentes; media historica...
3,Dupla contagem de um par que dispara mais de u...,Somar o mesmo par em capital imobilizado E inv...,Cada par recebe UMA acao primaria por preceden...,990 pares disparam mais de um sinal; todos ent...,A leitura por sinal (nao por acao primaria) po...


## Arquivos gerados

- `outputs/etapa7/recomendacoes_sku_loja.csv`
- `outputs/etapa7/recomendacoes_acao_universo.csv`
- `outputs/etapa7/recomendacoes_categoria_n1.csv`
- `outputs/etapa7/recomendacoes_lojas.csv`
- `outputs/etapa7/reprecificacao_candidatos.csv`
- `outputs/etapa7/priorizacao_acoes.csv`
- `outputs/etapa7/recomendacoes_melhoria.csv`
- `outputs/etapa7/validacoes_etapa7.csv`
- `outputs/etapa7/autoaudit_etapa7.csv`
- `outputs/etapa7/resumo_etapa7.md`
- `outputs/etapa7/documentacao_tecnica_etapa7.md`